# Player-course advantage — model walkthrough & backtest

> **Executive summary.** This notebook demonstrates a *transfer-learning* golf
> model: to estimate how much edge a player has on an **upcoming course** — before
> they've played it — we look at how they historically scored on holes that *look
> like* each of the target course's 18 holes. "Looks like" comes entirely from the
> already-built **v2.5** point-cloud similarity model; this layer only consumes it.
>
> The pipeline is: **v2.5 similar-hole sets → recency-weighted scorer → field
> ranking → diagnostics → leakage-guarded backtest → baselines → parameter sweep**.
> Every stage below calls the shipped `pipeline.modeling.player_course_advantage`
> modules — no model logic is re-implemented in the notebook.
>
> ⚠️ **Honesty note.** No real per-hole PGA scoring data is wired yet (see the
> [data-sourcing plan](../docs/player_course_advantage_data_sourcing.md), issue #46).
> **All numbers below are SYNTHETIC**, constructed to exercise the code paths — they
> are *not* evidence of predictive validity. The notebook auto-uses real data if a
> `GOLF_PCA_HISTORY_CSV` path is provided, otherwise it falls back to synthetic and
> says so.

## 0. Setup

Import the package. Runs fully offline — no internet, no Streamlit.

In [ ]:
import os, sys, tempfile
from pathlib import Path
import numpy as np, pandas as pd

# Put the repo root on the path whether run from repo root or notebooks/.
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, 'pipeline')):
    _p = os.path.dirname(_p)
if _p not in sys.path:
    sys.path.insert(0, _p)

from pipeline.modeling.player_course_advantage import (
    AdvantageParams, DEFAULT_PARAMS,
    load_similar_hole_sets, score_player_holes, explain_player_course,
    score_tournament_field, run_backtest, compare_baselines,
    model_beats_baselines, SweepGrid, run_sweep,
)
print('package imported from', _p)

## 1. The model in plain English

For a player `p` and a target hole `h` on the upcoming course `C`:

```
advantage(p, h) = weighted mean over (similar hole s, past occurrence o) of
                    similarity_weight(h, s) · recency_weight(year(o)) · outcome(p, s, o)

outcome = field_avg_score - player_score      # positive = beat the field
```

- **similarity_weight** — how close v2.5 says `s` is to `h` (normalized per target hole).
- **recency_weight** — `m ** age`, `age = (predict_season-1) - year`, so the prior
  season has weight 1.0 and older seasons fade.
- The 18 hole advantages aggregate to a course number (**sum** by default → expected
  strokes-vs-field over a round).

**Leakage guard:** only strictly-past seasons in the window are eligible,
`predict_season - W <= year < predict_season` — never the event season or later.

## 2. Synthetic data (clearly labelled)

We fabricate a small, valid dataset: a v2.5 similar-hole CSV (three candidate
courses per target hole) and a hole-score history for four players across several
seasons. Stronger players are given better field-adjusted outcomes so the model has
signal to find — this is a *demonstration*, not evidence.

In [ ]:
TARGET_COURSE = 'augusta_national'
N_HOLES = 9
CAND_COURSES = {'src_a': 0.5, 'src_b': 0.0, 'src_c': -0.5}  # course -> outcome delta
PLAYERS = {'p_ace': 2.0, 'p_good': 1.0, 'p_mid': 0.5, 'p_low': 0.0}  # base outcome
HIST_YEARS = range(2019, 2024)  # 2019..2023

def write_v25(root):
    rows = []
    for h in range(1, N_HOLES + 1):
        for rank, cand in enumerate(CAND_COURSES, start=1):
            rows.append(dict(target_hole_id=f'{TARGET_COURSE}:{h}',
                             candidate_hole_id=f'{cand}:{h}',
                             rank=rank, total_score=float(rank)))
    d = Path(root) / 'pointcloud_similarity' / 'baseline'
    d.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(d / 'similarity_results.csv', index=False)

def synthetic_history():
    rows, tid = [], 0
    for pid, base in PLAYERS.items():
        for cand, delta in CAND_COURSES.items():
            for yr in HIST_YEARS:
                for h in range(1, N_HOLES + 1):
                    outcome = base + delta
                    rows.append(dict(player_id=pid, player_name=pid.upper(),
                                     tournament_id=f'{yr}-{cand}-{tid}', year=yr, round=1,
                                     hole_number=h, course_slug=cand,
                                     hole_id_v25=f'{cand}:{h}', par=4,
                                     player_score=4 - outcome, field_avg_score=4))
                    tid += 1
    return pd.DataFrame(rows)

# Graceful degradation: prefer a real CSV if one is provided, else synthetic.
_real = os.environ.get('GOLF_PCA_HISTORY_CSV')
if _real and Path(_real).exists():
    history, DATA_SOURCE = pd.read_csv(_real), 'REAL'
else:
    history, DATA_SOURCE = synthetic_history(), 'SYNTHETIC'
print(f'DATA SOURCE: {DATA_SOURCE}  |  history rows: {len(history)}')
print('>>> All metrics below are', DATA_SOURCE, '- not a predictive-validity claim.' 
      if DATA_SOURCE == 'SYNTHETIC' else '- real data.')

## 3. Load v2.5 similar-hole sets

The loader turns the v2.5 CSV into normalized, per-target-hole similar-hole sets.

In [ ]:
_tmp = tempfile.mkdtemp()
write_v25(_tmp)
similar = load_similar_hole_sets(_tmp, TARGET_COURSE, top_n=3)
PARAMS = AdvantageParams(min_holes_covered=N_HOLES)  # small synthetic course
similar.head(6)

## 4. One player, per-hole breakdown + explanation

Score `p_ace` on the target course and explain *why*.

In [ ]:
per_hole = score_player_holes(history, similar, 'p_ace', TARGET_COURSE, 2024, PARAMS)
per_hole[['target_hole_id', 'hole_advantage', 'raw_occurrences', 'low_coverage']]

In [ ]:
expl = explain_player_course(history, similar, 'p_ace', TARGET_COURSE, 2024, params=PARAMS)
print('course advantage:', round(expl.course_summary['course_advantage'], 3),
      '| holes covered:', expl.course_summary['holes_covered'])
print('\nTop contributing target holes:')
display(expl.top_target_holes(3)[['target_hole_id', 'course_contribution']])
print('Top contributing similar holes:')
display(expl.top_similar_holes(3)[['target_hole_id', 'candidate_hole_id', 'weighted_contribution']])

## 5. Rank the whole field

Deterministic ranking — valid players first, low-coverage flagged.

In [ ]:
field = list(PLAYERS)
ranking = score_tournament_field(history, similar, field, TARGET_COURSE, 2024, params=PARAMS)
ranking[['rank', 'player_id', 'course_advantage', 'holes_covered', 'low_coverage']]

## 6. Retrospective backtest

Walk historical events; each is scored using **only** prior-season history
(leakage guard). We compare the predicted advantage to the (synthetic) actual
finish order.

In [ ]:
seasons = [2022, 2023, 2024]
finish = {pid: r for r, pid in enumerate(  # best base outcome -> rank 1
    sorted(PLAYERS, key=lambda p: PLAYERS[p], reverse=True), start=1)}
results = pd.concat([
    pd.DataFrame(dict(event_id=f'E{s}', predict_season=s, target_course_slug=TARGET_COURSE,
                      player_id=list(PLAYERS), finish_rank=[finish[p] for p in PLAYERS]))
    for s in seasons], ignore_index=True)

bt = run_backtest(history, similar, results, outcome_col='finish_rank',
                  higher_is_better=False, params=PARAMS, top_k=(2,))
print(bt.to_markdown())

## 7. Compare against simple baselines

Does the similar-hole model beat dumb baselines? Be honest — on this synthetic data it often just ties.

In [ ]:
comp = compare_baselines(history, similar, results, outcome_col='finish_rank',
                         higher_is_better=False, params=PARAMS, top_k=(2,))
display(comp[['ranker', 'spearman_pooled', 'coverage', 'n_pairs']])
print('Model strictly beats every baseline (spearman_pooled)?',
      model_beats_baselines(comp))

## 8. Parameter sensitivity (n, W, m)

Sweep the key knobs through the backtest and see which move the metric.

In [ ]:
def provider(config_name, top_n, weight_method):
    return load_similar_hole_sets(_tmp, TARGET_COURSE, config_name=config_name,
                                  top_n=top_n, weight_method=weight_method)

grid = SweepGrid(top_n=(1, 3), lookback_years=(3, 5), recency_decay=(0.7, 1.0),
                 min_holes_covered=(N_HOLES,))
sweep = run_sweep(history, results, provider, grid=grid, outcome_col='finish_rank',
                  higher_is_better=False, top_k=(2,))
display(sweep.table[['top_n', 'lookback_years', 'recency_decay',
                     'spearman_pooled', 'coverage', 'n_pairs']])
for w in sweep.warnings:
    print('WARNING:', w)

## 9. Limitations & data coverage

- **No real data.** Everything above is synthetic (`DATA_SOURCE` printed in §2). The
  model, backtest, baselines, and sweep are *plumbing validated on fabricated inputs*;
  they make **no** predictive claim. Real evaluation is blocked on a licensed per-hole
  source — see the [data-sourcing plan](../docs/player_course_advantage_data_sourcing.md) (#46).
- **Coverage gating is real.** Holes below `min_occurrences_per_hole` and players below
  `min_holes_covered` are withheld (never scored as 0), and travel with a `reason`.
- **Leakage is guarded but assumes clean inputs.** The window excludes the target
  season and later; it trusts that the upstream v2.5 similarity was itself built from
  pre-event data.
- **Defaults are uncalibrated.** `n`, `W`, `m`, and the coverage thresholds are sketch
  placeholders; the sweep (§8) is how they'd be tuned once real labels exist — with a
  train/validation split to avoid in-sample selection.

**Bottom line for a reviewer:** the full transfer-learning pipeline is implemented,
tested, leakage-guarded, and honest about coverage — it is ready to be pointed at real
hole-score data the moment that data is licensed and normalized.